# Activity 1. ETL - Metro CDMX

Carga y análisis general de la base de afluencia del Metro CDMX.

## 1. Cargar la base de datos

El archivo encontrado en esta carpeta se llama `afluenciastc_desglosado_06_2026.csv`. También se deja como candidato el nombre mencionado originalmente por si después agregas ese archivo al proyecto.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

candidatos_csv = [
    Path('afluenciasrc_desglosado_06_08_2026.csv'),
    Path('afluenciastc_desglosado_06_2026.csv'),
]

ruta_csv = next((ruta for ruta in candidatos_csv if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        'No se encontró el CSV esperado. Revisa que esté en la misma carpeta del notebook.'
    )

df = pd.read_csv(ruta_csv)

print(f'Archivo cargado: {ruta_csv}')
print(f'Filas: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]:,}')
df.head()

## 2. Revisión general

Se revisan duplicados exactos, datos nulos, tipos de datos y una descripción rápida de las columnas numéricas.

In [ ]:
resumen_general = pd.DataFrame({
    'tipo_dato': df.dtypes.astype(str),
    'nulos': df.isna().sum(),
    'nulos_%': (df.isna().mean() * 100).round(2),
    'valores_unicos': df.nunique(dropna=True),
})

duplicados_exactos = df.duplicated().sum()

print(f'Duplicados exactos: {duplicados_exactos:,}')
resumen_general

In [ ]:
df.describe(include='all').T

## 3. Formatos de columnas

`fecha` viene como texto al leer el CSV, por eso conviene convertirla a tipo fecha. También se verifica que `afluencia` sea numérica.

In [ ]:
df_limpio = df.copy()

df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'], errors='coerce')
df_limpio['afluencia'] = pd.to_numeric(df_limpio['afluencia'], errors='coerce')

print('Tipos después de convertir fecha y afluencia:')
display(df_limpio.dtypes)

print('Fechas inválidas después de convertir:', df_limpio['fecha'].isna().sum())
print('Afluencias inválidas después de convertir:', df_limpio['afluencia'].isna().sum())
print('Rango de fechas:', df_limpio['fecha'].min(), 'a', df_limpio['fecha'].max())

## 4. Valores raros por acentos

La columna `linea` trae valores con problemas de codificación, por ejemplo `LĂ­nea 1`, y también versiones sin acento como `Linea 1`. Primero se detectan esos casos y luego se normalizan.

In [ ]:
columnas_texto = df_limpio.select_dtypes(include='object').columns
patron_texto_raro = r'[\u00c3\u00c2\u0102\ufffd]'

reporte_texto_raro = []

for columna in columnas_texto:
    mascara = df_limpio[columna].astype(str).str.contains(patron_texto_raro, regex=True, na=False)
    reporte_texto_raro.append({
        'columna': columna,
        'filas_con_texto_raro': int(mascara.sum()),
        'ejemplos': sorted(df_limpio.loc[mascara, columna].dropna().unique())[:10],
    })

pd.DataFrame(reporte_texto_raro)

In [ ]:
print('Valores originales en linea:')
display(pd.Series(sorted(df_limpio['linea'].dropna().unique()), name='linea_original'))

print('Conteo original por linea:')
df_limpio['linea'].value_counts().sort_index()

## 5. Limpieza básica de texto

Esta función intenta corregir texto mal decodificado y después unifica `Linea` con `Línea` para que cada línea del Metro quede con un solo nombre.

In [ ]:
def corregir_mojibake(valor):
    if not isinstance(valor, str):
        return valor

    texto = valor
    for _ in range(3):
        try:
            corregido = texto.encode('latin1').decode('utf-8')
        except UnicodeError:
            break

        if corregido == texto:
            break
        texto = corregido

    return texto

for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].map(corregir_mojibake)

df_limpio['linea'] = df_limpio['linea'].str.replace(r'^Linea', 'L\u00ednea', regex=True)

homologacion_estaciones = {
    'G\u00f3mez Farias': 'G\u00f3mez Far\u00edas',
    'Pe\u00f1\u00f3n viejo': 'Pe\u00f1\u00f3n Viejo',
}

df_limpio['estacion'] = df_limpio['estacion'].replace(homologacion_estaciones)

print('Valores limpios en linea:')
display(pd.Series(sorted(df_limpio['linea'].dropna().unique()), name='linea_limpia'))

print('Estaciones homologadas:')
display(homologacion_estaciones)

print('Duplicados después de limpieza:', df_limpio.duplicated().sum())
print('Nulos después de limpieza:')
display(df_limpio.isna().sum())

## 6. Breve análisis general

Con la base revisada localmente:

- La base tiene **1,174,095 filas** y **7 columnas**.
- No se encontraron **duplicados exactos**.
- No se encontraron **datos nulos** en las columnas originales.
- `fecha` debe convertirse de texto a fecha; `anio` y `afluencia` ya se leen como enteros.
- `linea` tiene valores duplicados conceptualmente por codificación y acentos: aparecen versiones como `Linea 1`, `LĂ­nea 1` y, después de limpiar, deben quedar como `Línea 1`.
- La base cubre fechas de **2021-01-01** a **2026-06-30**.
- `tipo_pago` tiene tres categorías: `Boleto`, `Prepago` y `Gratuidad`.

In [ ]:
afluencia_por_linea = (
    df_limpio.groupby('linea', as_index=False)['afluencia']
    .sum()
    .sort_values('afluencia', ascending=False)
)

afluencia_por_linea

## 7. Revisión de cambios en nombres de estaciones

Una forma sencilla de revisar si cambiaron nombres de estaciones es comparar el catálogo de estaciones de cada línea a través del tiempo. Si una línea tiene siempre las mismas estaciones, entonces el catálogo debería repetirse en cada periodo.

Aquí lo revisamos de dos formas:

- Conteo de estaciones por `linea` y `anio`.
- Comparación exacta del conjunto de estaciones por `linea` y `anio` contra el catálogo completo observado para esa línea.

In [ ]:
conteo_estaciones_anio = (
    df_limpio.groupby(['linea', 'anio'])['estacion']
    .nunique()
    .reset_index(name='n_estaciones')
)

conteo_estaciones_anio.head(20)

In [ ]:
variacion_estaciones_anio = (
    conteo_estaciones_anio.groupby('linea')['n_estaciones']
    .agg(
        min_estaciones='min',
        max_estaciones='max',
        valores_distintos='nunique',
    )
    .reset_index()
)

lineas_con_cambio_conteo = variacion_estaciones_anio[
    variacion_estaciones_anio['valores_distintos'] > 1
]

print(f'Líneas con cambio en número de estaciones por año: {len(lineas_con_cambio_conteo)}')
lineas_con_cambio_conteo

El conteo ayuda, pero no es suficiente: podría haber cambiado un nombre y seguir existiendo el mismo número de estaciones. Por eso también comparamos el conjunto exacto de nombres.

In [ ]:
catalogo_estaciones_linea = (
    df_limpio.groupby('linea')['estacion']
    .agg(lambda valores: set(valores.dropna()))
)

reporte_cambios_estaciones = []

for (linea, anio), datos_periodo in df_limpio.groupby(['linea', 'anio']):
    estaciones_catalogo = catalogo_estaciones_linea.loc[linea]
    estaciones_periodo = set(datos_periodo['estacion'].dropna())

    faltantes = sorted(estaciones_catalogo - estaciones_periodo)
    adicionales = sorted(estaciones_periodo - estaciones_catalogo)

    if faltantes or adicionales:
        reporte_cambios_estaciones.append({
            'linea': linea,
            'anio': anio,
            'n_faltantes_vs_catalogo': len(faltantes),
            'faltantes_vs_catalogo': faltantes,
            'n_adicionales_vs_catalogo': len(adicionales),
            'adicionales_vs_catalogo': adicionales,
        })

reporte_cambios_estaciones = pd.DataFrame(reporte_cambios_estaciones)

print(f'Periodos con diferencias en el catálogo de estaciones: {len(reporte_cambios_estaciones)}')
reporte_cambios_estaciones

También podemos hacer la misma revisión por mes. Esto es más estricto porque detecta si alguna estación desaparece temporalmente del registro mensual.

In [ ]:
df_limpio['periodo_mes'] = df_limpio['fecha'].dt.to_period('M').astype(str)

conteo_estaciones_mes = (
    df_limpio.groupby(['linea', 'periodo_mes'])['estacion']
    .nunique()
    .reset_index(name='n_estaciones')
)

variacion_estaciones_mes = (
    conteo_estaciones_mes.groupby('linea')['n_estaciones']
    .agg(
        min_estaciones='min',
        max_estaciones='max',
        valores_distintos='nunique',
    )
    .reset_index()
)

lineas_con_cambio_mensual = variacion_estaciones_mes[
    variacion_estaciones_mes['valores_distintos'] > 1
]

print(f'Líneas con cambio en número de estaciones por mes: {len(lineas_con_cambio_mensual)}')
lineas_con_cambio_mensual

Ahora revisamos variantes de nombres. Para esto normalizamos temporalmente los nombres quitando acentos y pasando todo a minúsculas. Si dos nombres diferentes quedan iguales al normalizarse, probablemente son la misma estación escrita de dos maneras distintas.

In [ ]:
import unicodedata

def normalizar_texto_comparacion(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip().lower()
    texto = ''.join(
        caracter
        for caracter in unicodedata.normalize('NFD', texto)
        if unicodedata.category(caracter) != 'Mn'
    )
    return ' '.join(texto.split())

variantes_estacion = df_limpio.copy()
variantes_estacion['estacion_normalizada'] = variantes_estacion['estacion'].map(normalizar_texto_comparacion)

posibles_cambios_nombre = (
    variantes_estacion.groupby(['linea', 'estacion_normalizada'])['estacion']
    .agg(lambda valores: sorted(set(valores.dropna())))
    .reset_index(name='variantes_nombre')
)

posibles_cambios_nombre['n_variantes'] = posibles_cambios_nombre['variantes_nombre'].str.len()
posibles_cambios_nombre = posibles_cambios_nombre[
    posibles_cambios_nombre['n_variantes'] > 1
].sort_values(['linea', 'estacion_normalizada'])

posibles_cambios_nombre

**Conclusión de esta revisión:** las variantes `Gómez Farias` / `Gómez Farías` y `Peñón viejo` / `Peñón Viejo` ya fueron homologadas en `df_limpio`. Si la tabla anterior queda vacía, significa que ya no hay nombres de estaciones equivalentes escritos de dos formas distintas.

## 8. Llave primaria natural

La **llave primaria natural** es la combinación de columnas que debería identificar de forma única cada fila por el significado real de la tabla, sin crear un ID artificial.

En esta base, cada fila representa la afluencia de una combinación específica de:

- `fecha`
- `linea`
- `estacion`
- `tipo_pago`

Por eso, la llave natural propuesta es `fecha + linea + estacion + tipo_pago`. Si esa llave se cumple, no debería haber dos filas con la misma fecha, línea, estación y tipo de pago.

In [ ]:
llave_natural = ['fecha', 'linea', 'estacion', 'tipo_pago']

duplicados_llave_natural = df_limpio.duplicated(subset=llave_natural).sum()

resumen_llave_natural = pd.DataFrame({
    'llave_natural': [' + '.join(llave_natural)],
    'filas_totales': [len(df_limpio)],
    'combinaciones_unicas': [df_limpio[llave_natural].drop_duplicates().shape[0]],
    'duplicados_por_llave': [duplicados_llave_natural],
})

resumen_llave_natural

In [ ]:
if duplicados_llave_natural > 0:
    display(
        df_limpio[df_limpio.duplicated(subset=llave_natural, keep=False)]
        .sort_values(llave_natural)
        .head(20)
    )
else:
    print('La llave natural se cumple: no hay filas repetidas para fecha + linea + estacion + tipo_pago.')

## 9. Análisis de valores cero en `afluencia`

La base no tiene nulos, pero sí tiene muchos registros con `afluencia = 0`. No todos los ceros significan lo mismo. En esta sección vamos a revisar en qué líneas aparecen, graficarlos y separar dos casos importantes:

- **Cero total por estación-día:** la suma de `Boleto`, `Prepago` y `Gratuidad` es 0. Esto puede indicar cierre, suspensión, ausencia de operación o medición no comparable.
- **Cero parcial por tipo de pago:** una forma de pago está en 0, pero la estación sí tuvo afluencia total positiva ese día. Esto parece más relacionado con el producto de pago que con cierre de la estación.

In [ ]:
import matplotlib.pyplot as plt

df_ceros = df_limpio.copy()
df_ceros['es_cero'] = df_ceros['afluencia'].eq(0)
df_ceros['periodo_mes'] = df_ceros['fecha'].dt.to_period('M').astype(str)

resumen_ceros_linea = (
    df_ceros.groupby('linea')
    .agg(
        registros=('afluencia', 'size'),
        ceros=('es_cero', 'sum'),
        total_afluencia=('afluencia', 'sum'),
    )
    .reset_index()
)

resumen_ceros_linea['pct_ceros'] = (
    resumen_ceros_linea['ceros'] / resumen_ceros_linea['registros'] * 100
).round(2)

print(f"Total de registros con afluencia 0: {df_ceros['es_cero'].sum():,}")
print(f"Porcentaje total de ceros: {df_ceros['es_cero'].mean() * 100:.2f}%")

resumen_ceros_linea.sort_values('ceros', ascending=False)

In [ ]:
resumen_ceros_grafica = resumen_ceros_linea.sort_values('ceros', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(resumen_ceros_grafica['linea'], resumen_ceros_grafica['ceros'])
axes[0].set_title('Cantidad de registros con afluencia 0 por línea')
axes[0].set_xlabel('Línea')
axes[0].set_ylabel('Registros en cero')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(resumen_ceros_grafica['linea'], resumen_ceros_grafica['pct_ceros'])
axes[1].set_title('Porcentaje de registros en cero por línea')
axes[1].set_xlabel('Línea')
axes[1].set_ylabel('% de registros en cero')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Ahora se separa el problema por `tipo_pago`, porque muchos ceros pueden venir de un producto de pago que ya no se usa o que no aplica igual en todos los periodos.

In [ ]:
resumen_ceros_linea_pago = (
    df_ceros.groupby(['linea', 'tipo_pago'])
    .agg(
        registros=('afluencia', 'size'),
        ceros=('es_cero', 'sum'),
    )
    .reset_index()
)

resumen_ceros_linea_pago['pct_ceros'] = (
    resumen_ceros_linea_pago['ceros'] / resumen_ceros_linea_pago['registros'] * 100
).round(2)

resumen_ceros_linea_pago.sort_values('ceros', ascending=False).head(20)

In [ ]:
tabla_pct_ceros_pago = resumen_ceros_linea_pago.pivot(
    index='linea', columns='tipo_pago', values='pct_ceros'
).fillna(0)

fig, ax = plt.subplots(figsize=(9, 5))
imagen = ax.imshow(tabla_pct_ceros_pago, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(len(tabla_pct_ceros_pago.columns)))
ax.set_xticklabels(tabla_pct_ceros_pago.columns)
ax.set_yticks(range(len(tabla_pct_ceros_pago.index)))
ax.set_yticklabels(tabla_pct_ceros_pago.index)
ax.set_title('% de registros en cero por línea y tipo de pago')

for i, linea in enumerate(tabla_pct_ceros_pago.index):
    for j, tipo_pago in enumerate(tabla_pct_ceros_pago.columns):
        valor = tabla_pct_ceros_pago.loc[linea, tipo_pago]
        ax.text(j, i, f'{valor:.1f}%', ha='center', va='center', fontsize=8)

fig.colorbar(imagen, ax=ax, label='% de ceros')
plt.tight_layout()
plt.show()

Ahora revisamos si el cero fue de toda la estación en ese día o solo de un tipo de pago. Para eso se agrupa por `fecha`, `linea` y `estacion`.

In [ ]:
estacion_dia = (
    df_ceros.groupby(['fecha', 'linea', 'estacion'])
    .agg(
        afluencia_total_dia=('afluencia', 'sum'),
        tipos_pago_cero=('es_cero', 'sum'),
        tipos_pago=('tipo_pago', 'nunique'),
    )
    .reset_index()
)

estacion_dia['dia_total_cero'] = estacion_dia['afluencia_total_dia'].eq(0)
estacion_dia['dia_parcial_cero'] = (
    estacion_dia['tipos_pago_cero'].gt(0) & estacion_dia['afluencia_total_dia'].gt(0)
)

resumen_ceros_estacion_dia = (
    estacion_dia.groupby('linea')
    .agg(
        estacion_dias=('fecha', 'size'),
        dias_total_cero=('dia_total_cero', 'sum'),
        dias_parcial_cero=('dia_parcial_cero', 'sum'),
    )
    .reset_index()
)

resumen_ceros_estacion_dia['pct_dias_total_cero'] = (
    resumen_ceros_estacion_dia['dias_total_cero']
    / resumen_ceros_estacion_dia['estacion_dias']
    * 100
).round(2)

resumen_ceros_estacion_dia.sort_values('dias_total_cero', ascending=False)

In [ ]:
resumen_ceros_estacion_dia_grafica = resumen_ceros_estacion_dia.sort_values(
    'dias_total_cero', ascending=False
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(
    resumen_ceros_estacion_dia_grafica['linea'],
    resumen_ceros_estacion_dia_grafica['dias_total_cero'],
    label='Día total en cero',
)
ax.bar(
    resumen_ceros_estacion_dia_grafica['linea'],
    resumen_ceros_estacion_dia_grafica['dias_parcial_cero'],
    bottom=resumen_ceros_estacion_dia_grafica['dias_total_cero'],
    label='Día con cero parcial',
)

ax.set_title('Ceros totales y parciales por línea, a nivel estación-día')
ax.set_xlabel('Línea')
ax.set_ylabel('Cantidad de estación-días')
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.tight_layout()
plt.show()

Primer criterio de tratamiento:

- Si `afluencia_total_dia == 0` para una estación en una fecha, se marca como `cero_total_estacion_dia`. Para modelar afluencia real, ese valor es sospechoso y se crea `afluencia_modelo = NaN`.
- Si la estación tuvo afluencia total positiva, pero un tipo de pago está en 0, se marca como `cero_parcial_tipo_pago` y se conserva como 0, porque parece un cero real del producto de pago.

No borramos filas todavía; solo dejamos banderas para poder comparar después el modelo con y sin estas decisiones.

In [ ]:
df_limpio = df_limpio.merge(
    estacion_dia[['fecha', 'linea', 'estacion', 'afluencia_total_dia', 'dia_total_cero']],
    on=['fecha', 'linea', 'estacion'],
    how='left',
)

df_limpio['cero_total_estacion_dia'] = df_limpio['dia_total_cero']
df_limpio['cero_parcial_tipo_pago'] = (
    df_limpio['afluencia'].eq(0) & ~df_limpio['cero_total_estacion_dia']
)

df_limpio['afluencia_modelo'] = df_limpio['afluencia'].mask(
    df_limpio['cero_total_estacion_dia']
)

df_limpio = df_limpio.drop(columns=['dia_total_cero'])

resumen_tratamiento_ceros = pd.DataFrame({
    'registros': [len(df_limpio)],
    'ceros_originales': [df_limpio['afluencia'].eq(0).sum()],
    'registros_marcados_como_nan_para_modelo': [df_limpio['afluencia_modelo'].isna().sum()],
    'ceros_parciales_conservados': [df_limpio['cero_parcial_tipo_pago'].sum()],
})

resumen_tratamiento_ceros

**Conclusión preliminar:** los ceros no se concentran principalmente en `Línea A` y `Línea B`. En registros, las líneas con más ceros son `Línea 12` y `Línea 1`. `Línea A` y `Línea B` sí tienen muchos ceros en `Boleto`, pero casi no tienen días donde toda la estación sume cero. Por eso, para el modelo conviene tratar diferente los ceros totales de estación-día y los ceros parciales por tipo de pago.

## 10. Fechas y rachas de ceros totales

Para investigar si estos ceros coinciden con cierres o interrupciones reales, necesitamos listar fechas y detectar rachas consecutivas. Aquí se revisan dos niveles:

- **Línea-día:** qué porcentaje de estaciones de cada línea tuvo afluencia total 0 en cada fecha.
- **Estación-día:** qué estaciones tuvieron afluencia total 0 y por cuántos días seguidos.

In [ ]:
linea_dia_ceros = (
    estacion_dia.groupby(['fecha', 'linea'])
    .agg(
        estaciones_registradas=('estacion', 'nunique'),
        estaciones_total_cero=('dia_total_cero', 'sum'),
        afluencia_total_linea=('afluencia_total_dia', 'sum'),
    )
    .reset_index()
)

linea_dia_ceros['pct_estaciones_total_cero'] = (
    linea_dia_ceros['estaciones_total_cero']
    / linea_dia_ceros['estaciones_registradas']
    * 100
).round(2)

linea_dia_ceros['linea_dia_total_cero'] = linea_dia_ceros['afluencia_total_linea'].eq(0)

fechas_linea_total_cero = linea_dia_ceros[
    linea_dia_ceros['linea_dia_total_cero']
].sort_values(['linea', 'fecha'])

print(f'Línea-días donde toda la línea suma 0: {len(fechas_linea_total_cero):,}')
fechas_linea_total_cero.head(30)

In [ ]:
dias_mas_afectados = linea_dia_ceros[
    linea_dia_ceros['estaciones_total_cero'] > 0
].sort_values(
    ['pct_estaciones_total_cero', 'estaciones_total_cero'],
    ascending=False,
)

dias_mas_afectados.head(40)

La tabla anterior ayuda a buscar fechas puntuales. Para encontrar interrupciones largas, detectamos rachas consecutivas de ceros totales por estación.

In [ ]:
def detectar_rachas_consecutivas(datos, columnas_grupo, columna_fecha='fecha'):
    datos = datos.sort_values(columnas_grupo + [columna_fecha]).copy()

    datos['nueva_racha'] = (
        datos.groupby(columnas_grupo)[columna_fecha]
        .diff()
        .ne(pd.Timedelta(days=1))
    )

    datos['id_racha'] = datos.groupby(columnas_grupo)['nueva_racha'].cumsum()

    rachas = (
        datos.groupby(columnas_grupo + ['id_racha'])
        .agg(
            fecha_inicio=(columna_fecha, 'min'),
            fecha_fin=(columna_fecha, 'max'),
            dias=(columna_fecha, 'size'),
        )
        .reset_index()
        .drop(columns='id_racha')
    )

    rachas['semanas_aprox'] = (rachas['dias'] / 7).round(1)
    rachas['meses_aprox'] = (rachas['dias'] / 30.44).round(1)

    return rachas.sort_values('dias', ascending=False)

ceros_totales_estacion_dia = estacion_dia[
    estacion_dia['dia_total_cero']
].copy()

rachas_cero_estacion = detectar_rachas_consecutivas(
    ceros_totales_estacion_dia,
    columnas_grupo=['linea', 'estacion'],
)

rachas_cero_estacion.head(30)

In [ ]:
rachas_largas_cero_estacion = rachas_cero_estacion[
    rachas_cero_estacion['dias'] >= 7
].copy()

resumen_rachas_largas_linea = (
    rachas_largas_cero_estacion.groupby('linea')
    .agg(
        rachas_largas=('estacion', 'size'),
        estaciones_afectadas=('estacion', 'nunique'),
        max_dias_seguidos=('dias', 'max'),
        promedio_dias=('dias', 'mean'),
    )
    .reset_index()
)

resumen_rachas_largas_linea['promedio_dias'] = resumen_rachas_largas_linea['promedio_dias'].round(1)

resumen_rachas_largas_linea.sort_values('max_dias_seguidos', ascending=False)

In [ ]:
linea_interes = 'Línea 12'

rachas_cero_estacion[
    rachas_cero_estacion['linea'].eq(linea_interes)
].head(40)

También graficamos el porcentaje diario de estaciones en cero por línea. Los picos o bloques largos cerca de 100% son buenos candidatos para investigar como interrupciones estructurales.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for linea, datos_linea in linea_dia_ceros.groupby('linea'):
    ax.plot(
        datos_linea['fecha'],
        datos_linea['pct_estaciones_total_cero'],
        linewidth=1,
        alpha=0.8,
        label=linea,
    )

ax.set_title('Porcentaje diario de estaciones con afluencia total 0 por línea')
ax.set_xlabel('Fecha')
ax.set_ylabel('% de estaciones en cero total')
ax.legend(ncol=4, fontsize=8)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

**Lectura para el siguiente paso:** las rachas largas de `dia_total_cero` son candidatas a `structural breaks`. En especial, conviene contrastar las fechas de `Línea 12` y `Línea 1` contra noticias o comunicados de cierres/reaperturas, porque son las líneas con mayor afectación en ceros totales.

## 11. Contraste con eventos reales del Metro CDMX

Con las rachas largas detectadas, ahora se contrastan los datos contra eventos reales. Esto es importante para justificar si un cero debe excluirse, marcarse con bandera o modelarse.

Fuentes consultadas:

- Gobierno CDMX reportó el colapso de la estructura elevada de Línea 12 el **3 de mayo de 2021**, entre Olivos y San Lorenzo Tezonco.
- Gobierno CDMX reportó reapertura de Línea 12 por etapas: tramo subterráneo Mixcoac-Atlalilco el **15 de enero de 2023**, cinco estaciones elevadas Culhuacán-Periférico Oriente el **15 de julio de 2023**, y reapertura total con Tezonco-Tláhuac el **30 de enero de 2024**.
- Línea 1 tuvo cierres por modernización: Pantitlán-Salto del Agua desde **11 de julio de 2022**, reapertura Pantitlán-Isabel la Católica el **29 de octubre de 2023**, y cierre del tramo Salto del Agua-Observatorio desde la noche del **9 de noviembre de 2023**.
- Línea 9 cerró Pantitlán, Puebla y Ciudad Deportiva por renivelación desde **17 de diciembre de 2023** y reabrió el **10 de septiembre de 2024**.

In [ ]:
eventos_estructurales = pd.DataFrame([
    {
        'evento': 'Colapso y cierre de Línea 12',
        'linea': 'Línea 12',
        'fecha_inicio': '2021-05-04',
        'fecha_fin': '2023-01-14',
        'estaciones_evento': 'Todas / reapertura pendiente',
        'decision_etl': 'Marcar como structural break; ceros totales como ausentes para modelo',
    },
    {
        'evento': 'Reapertura tramo subterráneo L12',
        'linea': 'Línea 12',
        'fecha_inicio': '2023-01-15',
        'fecha_fin': '2023-07-14',
        'estaciones_evento': 'Mixcoac a Atlalilco operando; tramo elevado cerrado',
        'decision_etl': 'Marcar tramo elevado cerrado; conservar operación del tramo abierto',
    },
    {
        'evento': 'Reapertura parcial tramo elevado L12',
        'linea': 'Línea 12',
        'fecha_inicio': '2023-07-15',
        'fecha_fin': '2024-01-29',
        'estaciones_evento': 'Culhuacán a Periférico Oriente operando; Tezonco a Tláhuac cerrado',
        'decision_etl': 'Marcar estaciones aún cerradas como structural break',
    },
    {
        'evento': 'Reapertura total L12',
        'linea': 'Línea 12',
        'fecha_inicio': '2024-01-30',
        'fecha_fin': '2026-06-30',
        'estaciones_evento': 'Servicio restablecido en las 20 estaciones',
        'decision_etl': 'Periodo posterior comparable, salvo anomalías puntuales',
    },
    {
        'evento': 'Modernización L1 etapa 1',
        'linea': 'Línea 1',
        'fecha_inicio': '2022-07-11',
        'fecha_fin': '2023-10-28',
        'estaciones_evento': 'Pantitlán a Salto del Agua cerrado',
        'decision_etl': 'Marcar como structural break; ceros totales como ausentes para modelo',
    },
    {
        'evento': 'Modernización L1 etapa 2',
        'linea': 'Línea 1',
        'fecha_inicio': '2023-11-10',
        'fecha_fin': '2026-06-30',
        'estaciones_evento': 'Salto del Agua a Observatorio cerrado en el periodo de la base',
        'decision_etl': 'Marcar como structural break; revisar reaperturas parciales después',
    },
    {
        'evento': 'Renivelación tramo elevado L9',
        'linea': 'Línea 9',
        'fecha_inicio': '2023-12-17',
        'fecha_fin': '2024-09-09',
        'estaciones_evento': 'Pantitlán, Puebla y Ciudad Deportiva cerradas',
        'decision_etl': 'Marcar estaciones cerradas como structural break',
    },
])

eventos_estructurales['fecha_inicio'] = pd.to_datetime(eventos_estructurales['fecha_inicio'])
eventos_estructurales['fecha_fin'] = pd.to_datetime(eventos_estructurales['fecha_fin'])

eventos_estructurales

In [ ]:
comparacion_rachas_eventos = []

for _, racha in rachas_cero_estacion.iterrows():
    eventos_linea = eventos_estructurales[
        eventos_estructurales['linea'].eq(racha['linea'])
    ]

    for _, evento in eventos_linea.iterrows():
        inicio_overlap = max(racha['fecha_inicio'], evento['fecha_inicio'])
        fin_overlap = min(racha['fecha_fin'], evento['fecha_fin'])
        dias_overlap = (fin_overlap - inicio_overlap).days + 1

        if dias_overlap > 0:
            comparacion_rachas_eventos.append({
                'linea': racha['linea'],
                'estacion': racha['estacion'],
                'racha_inicio': racha['fecha_inicio'],
                'racha_fin': racha['fecha_fin'],
                'dias_racha': racha['dias'],
                'evento': evento['evento'],
                'evento_inicio': evento['fecha_inicio'],
                'evento_fin': evento['fecha_fin'],
                'dias_overlap': dias_overlap,
                'pct_racha_explicada': round(dias_overlap / racha['dias'] * 100, 2),
            })

comparacion_rachas_eventos = pd.DataFrame(comparacion_rachas_eventos)

comparacion_rachas_eventos.sort_values(
    ['dias_overlap', 'pct_racha_explicada'], ascending=False
).head(40)

In [ ]:
df_limpio['structural_break_metro'] = False
df_limpio['evento_estructural'] = pd.NA

for _, evento in eventos_estructurales.iterrows():
    mascara_evento = (
        df_limpio['linea'].eq(evento['linea'])
        & df_limpio['fecha'].between(evento['fecha_inicio'], evento['fecha_fin'])
    )

    df_limpio.loc[mascara_evento, 'structural_break_metro'] = True
    df_limpio.loc[mascara_evento, 'evento_estructural'] = evento['evento']

df_limpio['cero_total_sin_evento'] = (
    df_limpio['cero_total_estacion_dia']
    & ~df_limpio['structural_break_metro']
)

resumen_eventos_estructurales = (
    df_limpio.groupby(['linea', 'evento_estructural'], dropna=False)
    .agg(
        registros=('afluencia', 'size'),
        ceros=('afluencia', lambda s: s.eq(0).sum()),
        nulos_para_modelo=('afluencia_modelo', lambda s: s.isna().sum()),
    )
    .reset_index()
)

resumen_eventos_estructurales[
    resumen_eventos_estructurales['evento_estructural'].notna()
].sort_values(['linea', 'evento_estructural'])

Los ceros totales fuera de eventos estructurales conocidos se conservan en la base y se documentan con `cero_total_sin_evento`. No se eliminan todavía porque son evidencia de calidad de datos; para modelado se excluyen mediante `afluencia_modelo`, igual que los otros ceros totales.

In [ ]:
resumen_ceros_sin_evento = pd.DataFrame({
    'ceros_originales': [df_limpio['afluencia'].eq(0).sum()],
    'ceros_en_eventos_estructurales': [(
        df_limpio['afluencia'].eq(0) & df_limpio['structural_break_metro']
    ).sum()],
    'ceros_fuera_de_eventos': [(
        df_limpio['afluencia'].eq(0) & ~df_limpio['structural_break_metro']
    ).sum()],
    'ceros_totales_sin_evento': [df_limpio['cero_total_sin_evento'].sum()],
    'ceros_parciales_sin_evento': [(
        df_limpio['cero_parcial_tipo_pago'] & ~df_limpio['structural_break_metro']
    ).sum()],
})

resumen_ceros_sin_evento

In [ ]:
ceros_totales_sin_evento_por_linea = (
    df_limpio[df_limpio['cero_total_sin_evento']]
    .groupby(['linea', 'estacion'])
    .size()
    .reset_index(name='registros_cero_total_sin_evento')
    .sort_values('registros_cero_total_sin_evento', ascending=False)
)

ceros_totales_sin_evento_por_linea.head(30)

**Decisión ETL para structural breaks:** los cierres confirmados no se deben interpretar como demanda cero. Se conservan en la base original, pero para modelado se recomienda usar `afluencia_modelo` y las banderas `cero_total_estacion_dia` y `structural_break_metro`. Esto permite hacer el experimento pedido en el PDF: comparar el modelo con y sin las decisiones de structural breaks.

## 12. Validación contra catálogo oficial de estaciones

Como parte de la auditoría categórica, se compara el catálogo `linea-estacion` observado en la base contra el catálogo oficial del Metro CDMX. La referencia principal son las páginas oficiales de cada línea del STC Metro y el conjunto de Datos Abiertos de CDMX de líneas y estaciones.

Esta revisión ayuda a detectar estaciones mal asignadas a una línea, nombres antiguos, errores de acento o estaciones que aparecen en la base pero no corresponden a la red oficial.

In [ ]:
catalogo_oficial_metro = {
    'Línea 1': ['Observatorio', 'Tacubaya', 'Juanacatlán', 'Chapultepec', 'Sevilla', 'Insurgentes', 'Cuauhtémoc', 'Balderas', 'Salto del Agua', 'Isabel la Católica', 'Pino Suárez', 'Merced', 'Candelaria', 'San Lázaro', 'Moctezuma', 'Balbuena', 'Boulevard Puerto Aéreo', 'Gómez Farías', 'Zaragoza', 'Pantitlán'],
    'Línea 2': ['Cuatro Caminos', 'Panteones', 'Tacuba', 'Cuitláhuac', 'Popotla', 'Colegio Militar', 'Normal', 'San Cosme', 'Revolución', 'Hidalgo', 'Bellas Artes', 'Allende', 'Zócalo/Tenochtitlan', 'Pino Suárez', 'San Antonio Abad', 'Chabacano', 'Viaducto', 'Xola', 'Villa de Cortés', 'Nativitas', 'Portales', 'Ermita', 'General Anaya', 'Tasqueña'],
    'Línea 3': ['Indios Verdes', 'Deportivo 18 de Marzo', 'Potrero', 'La Raza', 'Tlatelolco', 'Guerrero', 'Hidalgo', 'Juárez', 'Balderas', 'Niños Héroes', 'Hospital General', 'Centro Médico', 'Etiopía/Plaza de la Transparencia', 'Eugenia', 'División del Norte', 'Zapata', 'Coyoacán', 'Viveros/Derechos Humanos', 'Miguel Ángel de Quevedo', 'Copilco', 'Universidad'],
    'Línea 4': ['Martín Carrera', 'Talismán', 'Bondojito', 'Consulado', 'Canal del Norte', 'Morelos', 'Candelaria', 'Fray Servando', 'Jamaica', 'Santa Anita'],
    'Línea 5': ['Pantitlán', 'Hangares', 'Terminal Aérea', 'Oceanía', 'Aragón', 'Eduardo Molina', 'Consulado', 'Valle Gómez', 'Misterios', 'La Raza', 'Autobuses del Norte', 'Instituto del Petróleo', 'Politécnico'],
    'Línea 6': ['El Rosario', 'Tezozómoc', 'UAM-Azcapotzalco', 'Ferrería/Arena Ciudad de México', 'Norte 45', 'Vallejo', 'Instituto del Petróleo', 'Lindavista', 'Deportivo 18 de Marzo', 'La Villa/Basílica', 'Martín Carrera'],
    'Línea 7': ['El Rosario', 'Aquiles Serdán', 'Camarones', 'Refinería', 'Tacuba', 'San Joaquín', 'Polanco', 'Auditorio', 'Constituyentes', 'Tacubaya', 'San Pedro de los Pinos', 'San Antonio', 'Mixcoac', 'Barranca del Muerto'],
    'Línea 8': ['Garibaldi/Lagunilla', 'Bellas Artes', 'San Juan de Letrán', 'Salto del Agua', 'Doctores', 'Obrera', 'Chabacano', 'La Viga', 'Santa Anita', 'Coyuya', 'Iztacalco', 'Apatlaco', 'Aculco', 'Escuadrón 201', 'Atlalilco', 'Iztapalapa', 'Cerro de la Estrella', 'UAM-I', 'Constitución de 1917'],
    'Línea 9': ['Pantitlán', 'Puebla', 'Ciudad Deportiva', 'Velódromo', 'Mixiuhca', 'Jamaica', 'Chabacano', 'Lázaro Cárdenas', 'Centro Médico', 'Chilpancingo', 'Patriotismo', 'Tacubaya'],
    'Línea A': ['Pantitlán', 'Agrícola Oriental', 'Canal de San Juan', 'Tepalcates', 'Guelatao', 'Peñón Viejo', 'Acatitla', 'Santa Marta', 'Los Reyes', 'La Paz'],
    'Línea B': ['Ciudad Azteca', 'Plaza Aragón', 'Olímpica', 'Ecatepec', 'Múzquiz', 'Río de los Remedios', 'Impulsora', 'Nezahualcóyotl', 'Villa de Aragón', 'Bosque de Aragón', 'Deportivo Oceanía', 'Oceanía', 'Romero Rubio', 'Ricardo Flores Magón', 'San Lázaro', 'Morelos', 'Tepito', 'Lagunilla', 'Garibaldi/Lagunilla', 'Guerrero', 'Buenavista'],
    'Línea 12': ['Mixcoac', 'Insurgentes Sur', 'Hospital 20 de Noviembre', 'Zapata', 'Parque de los Venados', 'Eje Central', 'Ermita', 'Mexicaltzingo', 'Atlalilco', 'Culhuacán', 'San Andrés Tomatlán', 'Lomas Estrella', 'Calle 11', 'Periférico Oriente', 'Tezonco', 'Olivos', 'Nopalera', 'Zapotitlán', 'Tlaltenco', 'Tláhuac'],
}

catalogo_oficial = pd.DataFrame(
    [(linea, estacion) for linea, estaciones in catalogo_oficial_metro.items() for estacion in estaciones],
    columns=['linea', 'estacion_oficial'],
)

catalogo_base = df_limpio[['linea', 'estacion']].drop_duplicates().copy()

catalogo_oficial['estacion_norm'] = catalogo_oficial['estacion_oficial'].map(normalizar_texto_comparacion)
catalogo_base['estacion_norm'] = catalogo_base['estacion'].map(normalizar_texto_comparacion)

comparacion_catalogo_oficial = catalogo_base.merge(
    catalogo_oficial,
    on=['linea', 'estacion_norm'],
    how='outer',
    indicator=True,
)

resumen_comparacion_catalogo = pd.DataFrame({
    'pares_linea_estacion_base': [len(catalogo_base)],
    'pares_linea_estacion_oficial': [len(catalogo_oficial)],
    'coinciden': [(comparacion_catalogo_oficial['_merge'] == 'both').sum()],
    'solo_en_base': [(comparacion_catalogo_oficial['_merge'] == 'left_only').sum()],
    'solo_en_catalogo_oficial': [(comparacion_catalogo_oficial['_merge'] == 'right_only').sum()],
})

resumen_comparacion_catalogo

In [ ]:
diferencias_catalogo = comparacion_catalogo_oficial[
    comparacion_catalogo_oficial['_merge'] != 'both'
].sort_values(['linea', '_merge', 'estacion', 'estacion_oficial'])

diferencias_catalogo

**Conclusión de catálogo:** si `solo_en_base` y `solo_en_catalogo_oficial` son 0, entonces todas las estaciones observadas en la base corresponden a la línea oficial del Metro CDMX. Si aparecen diferencias, se deben revisar como posibles errores de escritura, nombres oficiales distintos o estaciones asignadas a una línea incorrecta.

## 13. Análisis por tipo de pago

Ahora analizamos cómo se distribuye la afluencia por `tipo_pago`. Para evitar que los cierres o ausencias de operación distorsionen el análisis, usamos `afluencia_modelo`, que ya excluye los ceros totales de estación-día. Los ceros parciales por tipo de pago se conservan porque pueden representar cambios reales en el uso del producto de pago.

In [ ]:
df_pago = df_limpio.dropna(subset=['afluencia_modelo']).copy()

resumen_tipo_pago_total = (
    df_pago.groupby('tipo_pago')
    .agg(
        afluencia_total=('afluencia_modelo', 'sum'),
        registros=('afluencia_modelo', 'size'),
        promedio_por_registro=('afluencia_modelo', 'mean'),
    )
    .reset_index()
    .sort_values('afluencia_total', ascending=False)
)

resumen_tipo_pago_total['pct_afluencia'] = (
    resumen_tipo_pago_total['afluencia_total']
    / resumen_tipo_pago_total['afluencia_total'].sum()
    * 100
).round(2)

resumen_tipo_pago_total

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(
    resumen_tipo_pago_total['tipo_pago'],
    resumen_tipo_pago_total['afluencia_total'],
)

ax.set_title('Afluencia total por tipo de pago')
ax.set_xlabel('Tipo de pago')
ax.set_ylabel('Afluencia total')

for i, fila in resumen_tipo_pago_total.reset_index(drop=True).iterrows():
    ax.text(
        i,
        fila['afluencia_total'],
        f"{fila['pct_afluencia']:.1f}%",
        ha='center',
        va='bottom',
    )

plt.tight_layout()
plt.show()

Ahora revisamos la distribución por línea. Para comparar líneas de tamaño distinto, conviene usar una gráfica de barras apiladas al 100%, porque muestra la proporción de cada tipo de pago dentro de cada línea.

In [ ]:
afluencia_pago_linea = (
    df_pago.groupby(['linea', 'tipo_pago'])['afluencia_modelo']
    .sum()
    .reset_index()
)

tabla_pago_linea = afluencia_pago_linea.pivot(
    index='linea', columns='tipo_pago', values='afluencia_modelo'
).fillna(0)

orden_lineas_pago = tabla_pago_linea.sum(axis=1).sort_values(ascending=False).index
tabla_pago_linea = tabla_pago_linea.loc[orden_lineas_pago]

tabla_pago_linea_pct = tabla_pago_linea.div(tabla_pago_linea.sum(axis=1), axis=0) * 100

tabla_pago_linea_pct.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

tabla_pago_linea_pct.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    width=0.8,
)

ax.set_title('Distribución porcentual de afluencia por tipo de pago y línea')
ax.set_xlabel('Línea')
ax.set_ylabel('% de afluencia de la línea')
ax.legend(title='Tipo de pago', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Para estaciones no conviene graficar las 195 combinaciones `línea-estación` en una sola barra porque se vuelve ilegible. En su lugar usamos tres vistas:

- Tabla con el tipo de pago dominante por estación.
- Gráfica de las 30 estaciones con mayor afluencia.
- Boxplot de proporciones por tipo de pago para ver la distribución general entre estaciones.

In [ ]:
afluencia_pago_estacion = (
    df_pago.groupby(['linea', 'estacion', 'tipo_pago'])['afluencia_modelo']
    .sum()
    .reset_index()
)

tabla_pago_estacion = afluencia_pago_estacion.pivot_table(
    index=['linea', 'estacion'],
    columns='tipo_pago',
    values='afluencia_modelo',
    fill_value=0,
)

tabla_pago_estacion['afluencia_total'] = tabla_pago_estacion.sum(axis=1)

columnas_tipo_pago = [col for col in tabla_pago_estacion.columns if col != 'afluencia_total']
tabla_pago_estacion_pct = tabla_pago_estacion[columnas_tipo_pago].div(
    tabla_pago_estacion['afluencia_total'], axis=0
) * 100

tipo_pago_dominante_estacion = tabla_pago_estacion_pct.idxmax(axis=1)
pct_tipo_pago_dominante_estacion = tabla_pago_estacion_pct.max(axis=1).round(2)

resumen_pago_estacion = tabla_pago_estacion.reset_index()
resumen_pago_estacion['tipo_pago_dominante'] = tipo_pago_dominante_estacion.values
resumen_pago_estacion['pct_tipo_pago_dominante'] = pct_tipo_pago_dominante_estacion.values

resumen_pago_estacion.sort_values('afluencia_total', ascending=False).head(30)

In [ ]:
top_estaciones_pago = resumen_pago_estacion.sort_values(
    'afluencia_total', ascending=False
).head(30).copy()

top_estaciones_pago['linea_estacion'] = (
    top_estaciones_pago['linea'] + ' - ' + top_estaciones_pago['estacion']
)

top_estaciones_plot = top_estaciones_pago.set_index('linea_estacion')[columnas_tipo_pago]
top_estaciones_plot_pct = top_estaciones_plot.div(top_estaciones_plot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 10))
top_estaciones_plot_pct.sort_index().plot(kind='barh', stacked=True, ax=ax)

ax.set_title('Distribución porcentual por tipo de pago en las 30 estaciones con más afluencia')
ax.set_xlabel('% de afluencia de la estación')
ax.set_ylabel('Línea - estación')
ax.legend(title='Tipo de pago', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
proporciones_estacion_largo = (
    tabla_pago_estacion_pct.reset_index()
    .melt(
        id_vars=['linea', 'estacion'],
        value_vars=columnas_tipo_pago,
        var_name='tipo_pago',
        value_name='pct_afluencia_estacion',
    )
)

fig, ax = plt.subplots(figsize=(8, 5))
proporciones_estacion_largo.boxplot(
    column='pct_afluencia_estacion',
    by='tipo_pago',
    ax=ax,
)

ax.set_title('Distribución del porcentaje de afluencia por tipo de pago entre estaciones')
ax.set_xlabel('Tipo de pago')
ax.set_ylabel('% de afluencia dentro de cada estación')
plt.suptitle('')

plt.tight_layout()
plt.show()

proporciones_estacion_largo.groupby('tipo_pago')['pct_afluencia_estacion'].describe().round(2)

**Lectura inicial:** estas gráficas permiten ver si el modelo podría aprender patrones reales de demanda o solo diferencias contables entre productos de pago. Después de construir features de calendario, podremos revisar correlaciones con día de semana, mes, vacaciones/festivos y eventos estructurales.

## 14. Features de calendario: día de semana, fin de semana y mes

Construimos variables de calendario para analizar cuándo se usa más el Metro. Estas variables también servirán después como features para los modelos de regresión.

Para no mezclar cierres con demanda real, los agregados usan `afluencia_modelo`, que excluye los ceros totales de estación-día.

In [ ]:
dias_semana = {
    0: 'Lunes',
    1: 'Martes',
    2: 'Miércoles',
    3: 'Jueves',
    4: 'Viernes',
    5: 'Sábado',
    6: 'Domingo',
}

orden_dias_semana = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
orden_meses = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

df_limpio['dia_semana_num'] = df_limpio['fecha'].dt.dayofweek
df_limpio['dia_semana'] = df_limpio['dia_semana_num'].map(dias_semana)
df_limpio['es_fin_semana'] = df_limpio['dia_semana_num'].isin([5, 6])
df_limpio['numero_mes'] = df_limpio['fecha'].dt.month
df_limpio['nombre_mes'] = pd.Categorical(df_limpio['mes'], categories=orden_meses, ordered=True)
df_limpio['anio_mes'] = df_limpio['fecha'].dt.to_period('M').astype(str)

df_calendario = df_limpio.dropna(subset=['afluencia_modelo']).copy()

df_limpio[['fecha', 'dia_semana', 'es_fin_semana', 'nombre_mes', 'anio_mes']].head()

Primero revisamos el patrón total por día de semana. Usamos promedio diario para comparar días, porque la cantidad de lunes, martes, etc. no siempre es idéntica en el periodo.

In [ ]:
afluencia_diaria_total = (
    df_calendario.groupby('fecha')['afluencia_modelo']
    .sum()
    .reset_index(name='afluencia_diaria')
)

afluencia_diaria_total['dia_semana'] = afluencia_diaria_total['fecha'].dt.dayofweek.map(dias_semana)
afluencia_diaria_total['es_fin_semana'] = afluencia_diaria_total['fecha'].dt.dayofweek.isin([5, 6])
afluencia_diaria_total['nombre_mes'] = pd.Categorical(
    afluencia_diaria_total['fecha'].dt.month_name(locale=None),
    ordered=False,
)

resumen_dia_semana_total = (
    afluencia_diaria_total.groupby('dia_semana')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reindex(orden_dias_semana)
    .reset_index()
)

resumen_dia_semana_total[['mean', 'median', 'min', 'max']] = resumen_dia_semana_total[['mean', 'median', 'min', 'max']].round(0)
resumen_dia_semana_total

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(resumen_dia_semana_total['dia_semana'], resumen_dia_semana_total['mean'])
ax.set_title('Afluencia promedio diaria por día de semana')
ax.set_xlabel('Día de semana')
ax.set_ylabel('Afluencia promedio diaria')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
resumen_fin_semana = (
    afluencia_diaria_total.groupby('es_fin_semana')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

resumen_fin_semana['tipo_dia'] = resumen_fin_semana['es_fin_semana'].map({False: 'Entre semana', True: 'Fin de semana'})
resumen_fin_semana[['mean', 'median', 'min', 'max']] = resumen_fin_semana[['mean', 'median', 'min', 'max']].round(0)

resumen_fin_semana[['tipo_dia', 'mean', 'median', 'min', 'max', 'count']]

Ahora se revisa el patrón por línea y día de semana. Esto permite saber qué líneas son más usadas en cada tipo de día.

In [ ]:
afluencia_linea_dia = (
    df_calendario.groupby(['fecha', 'linea'])['afluencia_modelo']
    .sum()
    .reset_index(name='afluencia_linea_dia')
)

afluencia_linea_dia['dia_semana'] = afluencia_linea_dia['fecha'].dt.dayofweek.map(dias_semana)

tabla_linea_dia_semana = (
    afluencia_linea_dia.groupby(['linea', 'dia_semana'])['afluencia_linea_dia']
    .mean()
    .reset_index()
    .pivot(index='linea', columns='dia_semana', values='afluencia_linea_dia')
    .reindex(columns=orden_dias_semana)
)

tabla_linea_dia_semana.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
imagen = ax.imshow(tabla_linea_dia_semana, aspect='auto', cmap='YlGnBu')

ax.set_xticks(range(len(tabla_linea_dia_semana.columns)))
ax.set_xticklabels(tabla_linea_dia_semana.columns, rotation=30)
ax.set_yticks(range(len(tabla_linea_dia_semana.index)))
ax.set_yticklabels(tabla_linea_dia_semana.index)
ax.set_title('Afluencia promedio por línea y día de semana')

fig.colorbar(imagen, ax=ax, label='Afluencia promedio')
plt.tight_layout()
plt.show()

Para estaciones buscamos las combinaciones `línea-estación-día` con mayor afluencia promedio. Esta tabla ayuda a detectar qué estaciones son más concurridas y en qué día suelen tener más usuarios.

In [ ]:
afluencia_estacion_dia = (
    df_calendario.groupby(['fecha', 'linea', 'estacion'])['afluencia_modelo']
    .sum()
    .reset_index(name='afluencia_estacion_dia')
)

afluencia_estacion_dia['dia_semana'] = afluencia_estacion_dia['fecha'].dt.dayofweek.map(dias_semana)

promedio_estacion_dia_semana = (
    afluencia_estacion_dia.groupby(['linea', 'estacion', 'dia_semana'])['afluencia_estacion_dia']
    .mean()
    .reset_index()
)

top_estacion_dia_semana = promedio_estacion_dia_semana.sort_values(
    'afluencia_estacion_dia', ascending=False
).head(30)

top_estacion_dia_semana

In [ ]:
top_estacion_dia_semana_plot = top_estacion_dia_semana.copy()
top_estacion_dia_semana_plot['linea_estacion_dia'] = (
    top_estacion_dia_semana_plot['linea']
    + ' - '
    + top_estacion_dia_semana_plot['estacion']
    + ' - '
    + top_estacion_dia_semana_plot['dia_semana']
)

fig, ax = plt.subplots(figsize=(11, 9))
ax.barh(
    top_estacion_dia_semana_plot['linea_estacion_dia'][::-1],
    top_estacion_dia_semana_plot['afluencia_estacion_dia'][::-1],
)

ax.set_title('Top 30 estaciones-día con mayor afluencia promedio')
ax.set_xlabel('Afluencia promedio')
ax.set_ylabel('Línea - estación - día')

plt.tight_layout()
plt.show()

Finalmente revisamos el patrón mensual. Para evitar que un mes con más días parezca mayor solo por duración, se usa afluencia promedio diaria por mes.

In [ ]:
afluencia_diaria_total['numero_mes'] = afluencia_diaria_total['fecha'].dt.month
afluencia_diaria_total['nombre_mes'] = pd.Categorical(
    afluencia_diaria_total['numero_mes'].map(dict(enumerate(orden_meses, start=1))),
    categories=orden_meses,
    ordered=True,
)

resumen_mes_total = (
    afluencia_diaria_total.groupby('nombre_mes', observed=True)['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

resumen_mes_total[['mean', 'median', 'min', 'max']] = resumen_mes_total[['mean', 'median', 'min', 'max']].round(0)
resumen_mes_total

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(resumen_mes_total['nombre_mes'].astype(str), resumen_mes_total['mean'])
ax.set_title('Afluencia promedio diaria por mes')
ax.set_xlabel('Mes')
ax.set_ylabel('Afluencia promedio diaria')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**Lectura inicial:** estas variables de calendario permiten comparar demanda por día de semana, fin de semana y mes. Después podemos usarlas como features del modelo y medir si realmente ayudan contra un baseline.

## 15. Media, mediana y total por estación

La media y la mediana cuentan cosas distintas:

- **Media:** promedio diario. Es útil para estimar volumen esperado, pero puede subir mucho si hay picos extremos.
- **Mediana:** día típico. Es menos sensible a picos, pero puede ocultar eventos de alta demanda.
- **Total:** volumen acumulado. Favorece estaciones con más días operando y periodos más completos.

Para este análisis, la mediana no reemplaza a la media: la usamos como contraste para saber si las estaciones top son consistentemente altas o si aparecen arriba por pocos días atípicos.

In [ ]:
resumen_estacion_media_mediana = (
    afluencia_estacion_dia.groupby(['linea', 'estacion'])['afluencia_estacion_dia']
    .agg(
        dias_observados='count',
        total='sum',
        media='mean',
        mediana='median',
        p25=lambda s: s.quantile(0.25),
        p75=lambda s: s.quantile(0.75),
        maximo='max',
    )
    .reset_index()
)

resumen_estacion_media_mediana['media_mediana_ratio'] = (
    resumen_estacion_media_mediana['media']
    / resumen_estacion_media_mediana['mediana'].replace(0, pd.NA)
)

columnas_redondear = ['total', 'media', 'mediana', 'p25', 'p75', 'maximo', 'media_mediana_ratio']
resumen_estacion_media_mediana[columnas_redondear] = resumen_estacion_media_mediana[columnas_redondear].round(2)

resumen_estacion_media_mediana.sort_values('media', ascending=False).head(20)

In [ ]:
top_por_mediana = resumen_estacion_media_mediana.sort_values('mediana', ascending=False).head(20)
top_por_mediana

In [ ]:
comparacion_top_media_mediana = resumen_estacion_media_mediana.copy()
comparacion_top_media_mediana['rank_media'] = comparacion_top_media_mediana['media'].rank(ascending=False, method='min')
comparacion_top_media_mediana['rank_mediana'] = comparacion_top_media_mediana['mediana'].rank(ascending=False, method='min')
comparacion_top_media_mediana['diferencia_rank'] = (
    comparacion_top_media_mediana['rank_mediana'] - comparacion_top_media_mediana['rank_media']
)

comparacion_top_media_mediana.sort_values('rank_media').head(30)[[
    'linea', 'estacion', 'dias_observados', 'media', 'mediana',
    'media_mediana_ratio', 'rank_media', 'rank_mediana', 'diferencia_rank'
]]

In [ ]:
estaciones_media_muy_superior_mediana = resumen_estacion_media_mediana[
    resumen_estacion_media_mediana['media_mediana_ratio'] >= 1.25
].sort_values('media_mediana_ratio', ascending=False)

estaciones_media_muy_superior_mediana.head(30)

In [ ]:
top_comparacion = resumen_estacion_media_mediana.sort_values('media', ascending=False).head(20).copy()
top_comparacion['linea_estacion'] = top_comparacion['linea'] + ' - ' + top_comparacion['estacion']

fig, ax = plt.subplots(figsize=(11, 8))

y = range(len(top_comparacion))
ax.barh(y, top_comparacion['media'], label='Media', alpha=0.8)
ax.scatter(top_comparacion['mediana'], y, color='black', label='Mediana', zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(top_comparacion['linea_estacion'])
ax.invert_yaxis()
ax.set_title('Top 20 estaciones: media diaria vs mediana diaria')
ax.set_xlabel('Afluencia diaria')
ax.set_ylabel('Línea - estación')
ax.legend()

plt.tight_layout()
plt.show()

**Interpretación:** si una estación aparece arriba tanto por media como por mediana, su alta demanda es consistente. Si la media es mucho mayor que la mediana, hay picos o días atípicos que elevan el promedio. Para demanda típica, la mediana es útil; para planeación de capacidad, también importan la media, el percentil 75 y el máximo.

## 16. Estaciones donde media y mediana difieren mucho

Ahora buscamos estaciones donde la media y la mediana cuentan historias distintas. Esto puede revelar:

- **Media mucho mayor que mediana:** pocos días muy altos elevan el promedio. Pueden ser eventos, partidos, conciertos o días especiales.
- **Mediana mucho mayor que media:** hay muchos días bajos que jalan el promedio hacia abajo. Pueden ser domingos, festivos, cierres parciales, periodos de recuperación o estacionalidad.

Esta revisión es exploratoria: identifica candidatos que después se pueden contrastar con eventos externos.

In [ ]:
analisis_media_mediana = resumen_estacion_media_mediana.copy()
analisis_media_mediana['diferencia_media_mediana'] = (
    analisis_media_mediana['media'] - analisis_media_mediana['mediana']
)
analisis_media_mediana['diferencia_pct_vs_mediana'] = (
    analisis_media_mediana['diferencia_media_mediana']
    / analisis_media_mediana['mediana'].replace(0, pd.NA)
    * 100
).round(2)

media_mayor_que_mediana = analisis_media_mediana.sort_values(
    'diferencia_pct_vs_mediana', ascending=False
).head(20)

mediana_mayor_que_media = analisis_media_mediana.sort_values(
    'diferencia_pct_vs_mediana', ascending=True
).head(20)

media_mayor_que_mediana[[
    'linea', 'estacion', 'dias_observados', 'media', 'mediana',
    'diferencia_media_mediana', 'diferencia_pct_vs_mediana', 'p75', 'maximo'
]]

In [ ]:
mediana_mayor_que_media[[
    'linea', 'estacion', 'dias_observados', 'media', 'mediana',
    'diferencia_media_mediana', 'diferencia_pct_vs_mediana', 'p25', 'maximo'
]]

Creamos una función para analizar una estación específica. Devuelve:

- Resumen general.
- Perfil por día de semana.
- Perfil por mes.
- Fechas con mayor afluencia.
- Fechas con menor afluencia positiva.

Esto nos ayuda a inferir si la diferencia se debe a picos altos, días bajos recurrentes o patrones semanales.

In [ ]:
def perfil_estacion(linea, estacion, n_fechas=10):
    datos = afluencia_estacion_dia[
        afluencia_estacion_dia['linea'].eq(linea)
        & afluencia_estacion_dia['estacion'].eq(estacion)
    ].copy()

    if datos.empty:
        raise ValueError(f'No hay datos para {linea} - {estacion}')

    datos['dia_semana'] = datos['fecha'].dt.dayofweek.map(dias_semana)
    datos['nombre_mes'] = pd.Categorical(
        datos['fecha'].dt.month.map(dict(enumerate(orden_meses, start=1))),
        categories=orden_meses,
        ordered=True,
    )

    resumen = datos['afluencia_estacion_dia'].agg(
        dias_observados='count',
        total='sum',
        media='mean',
        mediana='median',
        p25=lambda s: s.quantile(0.25),
        p75=lambda s: s.quantile(0.75),
        maximo='max',
    ).to_frame().T.round(2)

    resumen['linea'] = linea
    resumen['estacion'] = estacion
    resumen['media_mediana_ratio'] = (
        resumen['media'] / resumen['mediana'].replace(0, pd.NA)
    ).round(2)

    perfil_dia = (
        datos.groupby('dia_semana')['afluencia_estacion_dia']
        .agg(['mean', 'median', 'min', 'max', 'count'])
        .reindex(orden_dias_semana)
        .round(0)
    )

    perfil_mes = (
        datos.groupby('nombre_mes', observed=True)['afluencia_estacion_dia']
        .agg(['mean', 'median', 'min', 'max', 'count'])
        .reindex(orden_meses)
        .round(0)
    )

    fechas_altas = datos.sort_values('afluencia_estacion_dia', ascending=False).head(n_fechas)
    fechas_bajas = datos[datos['afluencia_estacion_dia'].gt(0)].sort_values('afluencia_estacion_dia').head(n_fechas)

    return resumen, perfil_dia, perfil_mes, fechas_altas, fechas_bajas

resumen_velodromo, perfil_dia_velodromo, perfil_mes_velodromo, fechas_altas_velodromo, fechas_bajas_velodromo = perfil_estacion(
    'Línea 9', 'Velódromo'
)

resumen_velodromo

In [ ]:
perfil_dia_velodromo

In [ ]:
fechas_altas_velodromo[['fecha', 'linea', 'estacion', 'dia_semana', 'afluencia_estacion_dia']]

En `Velódromo`, la media es mucho mayor que la mediana. Eso sugiere que la estación tiene días pico que no representan el comportamiento típico. Por su ubicación, es una candidata natural para revisar eventos masivos cercanos, pero esa explicación debe validarse con una fuente externa antes de afirmarla en el reporte.

In [ ]:
estaciones_a_revisar = pd.concat([
    media_mayor_que_mediana.head(5),
    mediana_mayor_que_media.head(5),
], ignore_index=True)[['linea', 'estacion']].drop_duplicates()

perfiles_estaciones_revisar = []

for _, fila in estaciones_a_revisar.iterrows():
    resumen, perfil_dia, perfil_mes, fechas_altas, fechas_bajas = perfil_estacion(
        fila['linea'], fila['estacion'], n_fechas=5
    )
    perfiles_estaciones_revisar.append(resumen)

perfiles_estaciones_revisar = pd.concat(perfiles_estaciones_revisar, ignore_index=True)
perfiles_estaciones_revisar[[
    'linea', 'estacion', 'dias_observados', 'media', 'mediana',
    'media_mediana_ratio', 'p25', 'p75', 'maximo'
]].sort_values('media_mediana_ratio', ascending=False)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

media_mayor_que_mediana.head(10).sort_values('diferencia_pct_vs_mediana').plot(
    kind='barh',
    x='estacion',
    y='diferencia_pct_vs_mediana',
    ax=axes[0],
    legend=False,
)
axes[0].set_title('Media mayor que mediana: posibles picos altos')
axes[0].set_xlabel('Diferencia % vs mediana')
axes[0].set_ylabel('Estación')

mediana_mayor_que_media.head(10).assign(
    diferencia_abs_pct=lambda d: d['diferencia_pct_vs_mediana'].abs()
).sort_values('diferencia_abs_pct').plot(
    kind='barh',
    x='estacion',
    y='diferencia_abs_pct',
    ax=axes[1],
    legend=False,
)
axes[1].set_title('Mediana mayor que media: posibles días bajos recurrentes')
axes[1].set_xlabel('Diferencia % absoluta vs mediana')
axes[1].set_ylabel('Estación')

plt.tight_layout()
plt.show()

**Lectura inicial:** `Velódromo` es el caso más claro de media mayor que mediana, por lo que probablemente tiene picos de demanda en fechas específicas. En cambio, estaciones donde la mediana supera mucho a la media parecen tener varios días bajos que reducen el promedio. Para explicar causalmente esos patrones se requiere cruzar con eventos externos, festivos, calendario escolar o eventos cercanos.

## 17. Festivos de México y afluencia

El PDF pide unir festivos mexicanos. En esta sección se construye una tabla de días festivos oficiales para 2021-2026 y se cruza contra la afluencia.

La idea es revisar tres cosas:

- Si los festivos tienen menor o mayor afluencia que un día normal.
- Si los picos de afluencia coinciden con festivos.
- Si las estaciones donde media y mediana difieren mucho tienen patrones relacionados con festivos.

In [ ]:
def primer_lunes(anio, mes):
    fechas = pd.date_range(f'{anio}-{mes:02d}-01', periods=7, freq='D')
    return fechas[fechas.dayofweek == 0][0]

def tercer_lunes(anio, mes):
    fechas = pd.date_range(f'{anio}-{mes:02d}-01', periods=21, freq='D')
    return fechas[fechas.dayofweek == 0][2]

festivos_mexico = []

for anio in range(df_limpio['fecha'].dt.year.min(), df_limpio['fecha'].dt.year.max() + 1):
    festivos_mexico.extend([
        {'fecha': pd.Timestamp(f'{anio}-01-01'), 'festivo': 'Año Nuevo'},
        {'fecha': primer_lunes(anio, 2), 'festivo': 'Constitución Mexicana'},
        {'fecha': tercer_lunes(anio, 3), 'festivo': 'Natalicio Benito Juárez'},
        {'fecha': pd.Timestamp(f'{anio}-05-01'), 'festivo': 'Día del Trabajo'},
        {'fecha': pd.Timestamp(f'{anio}-09-16'), 'festivo': 'Independencia de México'},
        {'fecha': tercer_lunes(anio, 11), 'festivo': 'Revolución Mexicana'},
        {'fecha': pd.Timestamp(f'{anio}-12-25'), 'festivo': 'Navidad'},
    ])

festivos_mexico = pd.DataFrame(festivos_mexico)
festivos_mexico = festivos_mexico[
    festivos_mexico['fecha'].between(df_limpio['fecha'].min(), df_limpio['fecha'].max())
].sort_values('fecha')

festivos_mexico.head(15)

In [ ]:
df_limpio = df_limpio.merge(festivos_mexico, on='fecha', how='left')
df_limpio['es_festivo'] = df_limpio['festivo'].notna()
df_limpio['festivo'] = df_limpio['festivo'].fillna('No festivo')

df_calendario = df_limpio.dropna(subset=['afluencia_modelo']).copy()

afluencia_diaria_total_festivos = (
    df_calendario.groupby(['fecha', 'es_festivo', 'festivo'])['afluencia_modelo']
    .sum()
    .reset_index(name='afluencia_diaria')
)

afluencia_diaria_total_festivos['dia_semana'] = afluencia_diaria_total_festivos['fecha'].dt.dayofweek.map(dias_semana)

resumen_festivo_total = (
    afluencia_diaria_total_festivos.groupby('es_festivo')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

resumen_festivo_total['tipo_dia'] = resumen_festivo_total['es_festivo'].map({False: 'No festivo', True: 'Festivo'})
resumen_festivo_total[['mean', 'median', 'min', 'max']] = resumen_festivo_total[['mean', 'median', 'min', 'max']].round(0)

resumen_festivo_total[['tipo_dia', 'mean', 'median', 'min', 'max', 'count']]

In [ ]:
resumen_por_festivo = (
    afluencia_diaria_total_festivos[afluencia_diaria_total_festivos['es_festivo']]
    .groupby('festivo')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)

resumen_por_festivo[['mean', 'median', 'min', 'max']] = resumen_por_festivo[['mean', 'median', 'min', 'max']].round(0)
resumen_por_festivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(resumen_festivo_total['tipo_dia'], resumen_festivo_total['mean'])
axes[0].set_title('Afluencia promedio diaria: festivo vs no festivo')
axes[0].set_xlabel('Tipo de día')
axes[0].set_ylabel('Afluencia promedio diaria')

axes[1].barh(resumen_por_festivo['festivo'], resumen_por_festivo['mean'])
axes[1].set_title('Afluencia promedio diaria por festivo')
axes[1].set_xlabel('Afluencia promedio diaria')
axes[1].set_ylabel('Festivo')

plt.tight_layout()
plt.show()

Ahora revisamos si los días con mayor afluencia total coinciden con festivos. Si los picos no son festivos, pueden deberse más a días laborales, regreso a actividades, eventos locales o patrones de movilidad ordinaria.

In [ ]:
top_dias_afluencia_total = afluencia_diaria_total_festivos.sort_values(
    'afluencia_diaria', ascending=False
).head(30)

top_dias_baja_afluencia_total = afluencia_diaria_total_festivos.sort_values(
    'afluencia_diaria', ascending=True
).head(30)

top_dias_afluencia_total

In [ ]:
top_dias_baja_afluencia_total

También analizamos estaciones atípicas contra festivos: las que tienen media mucho mayor que mediana y las que tienen mediana mucho mayor que media.

In [ ]:
estaciones_extremas_media_mediana = pd.concat([
    media_mayor_que_mediana.head(10).assign(tipo_diferencia='Media mayor que mediana'),
    mediana_mayor_que_media.head(10).assign(tipo_diferencia='Mediana mayor que media'),
], ignore_index=True)[['linea', 'estacion', 'tipo_diferencia']].drop_duplicates()

afluencia_estacion_dia_festivos = afluencia_estacion_dia.merge(
    festivos_mexico,
    on='fecha',
    how='left',
)
afluencia_estacion_dia_festivos['es_festivo'] = afluencia_estacion_dia_festivos['festivo'].notna()
afluencia_estacion_dia_festivos['festivo'] = afluencia_estacion_dia_festivos['festivo'].fillna('No festivo')

afluencia_estacion_dia_festivos = afluencia_estacion_dia_festivos.merge(
    estaciones_extremas_media_mediana,
    on=['linea', 'estacion'],
    how='inner',
)

comparacion_festivos_estaciones_extremas = (
    afluencia_estacion_dia_festivos.groupby(['tipo_diferencia', 'linea', 'estacion', 'es_festivo'])['afluencia_estacion_dia']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

comparacion_festivos_estaciones_extremas['tipo_dia'] = comparacion_festivos_estaciones_extremas['es_festivo'].map({False: 'No festivo', True: 'Festivo'})
comparacion_festivos_estaciones_extremas[['mean', 'median', 'min', 'max']] = comparacion_festivos_estaciones_extremas[['mean', 'median', 'min', 'max']].round(0)

comparacion_festivos_estaciones_extremas.sort_values(['tipo_diferencia', 'linea', 'estacion', 'es_festivo'])

In [ ]:
fechas_altas_extremas = (
    afluencia_estacion_dia_festivos.sort_values('afluencia_estacion_dia', ascending=False)
    .groupby(['tipo_diferencia', 'linea', 'estacion'])
    .head(5)
    .sort_values(['tipo_diferencia', 'linea', 'estacion', 'afluencia_estacion_dia'], ascending=[True, True, True, False])
)

fechas_altas_extremas[['tipo_diferencia', 'fecha', 'linea', 'estacion', 'afluencia_estacion_dia', 'es_festivo', 'festivo']]

**Lectura inicial:** los festivos suelen tener menor afluencia que los días no festivos. Si los picos de estaciones atípicas no caen en festivos, entonces probablemente se deben a eventos locales, patrones laborales/escolares o demanda ordinaria en días específicos. Esto ayuda a separar efectos de calendario nacional de efectos locales de cada estación.

## 18. Clima diario en CDMX

El PDF pide unir clima diario y auditar si aporta información. Para hacerlo de forma reproducible se usa la API histórica de Open-Meteo, que permite descargar variables diarias sin API key.

Referencia: Open-Meteo Historical Weather API, endpoint `/v1/archive`, con variables diarias como temperatura media, precipitación, horas de lluvia y viento.

Coordenadas usadas para CDMX centro: latitud `19.4326`, longitud `-99.1332`.

In [ ]:
from http.client import IncompleteRead
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen
import json
import time

ruta_clima = ruta_csv.parent / 'clima_cdmx_open_meteo_2021_2026.csv'

variables_clima = [
    'temperature_2m_mean',
    'temperature_2m_max',
    'temperature_2m_min',
    'precipitation_sum',
    'rain_sum',
    'precipitation_hours',
    'wind_speed_10m_max',
    'wind_gusts_10m_max',
    'weather_code',
]

def descargar_json_con_reintentos(url, reintentos=5, espera_segundos=3):
    for intento in range(1, reintentos + 1):
        try:
            with urlopen(url, timeout=60) as respuesta:
                contenido = respuesta.read()
            return json.loads(contenido.decode('utf-8'))
        except (IncompleteRead, URLError, TimeoutError, ConnectionError) as error:
            if intento == reintentos:
                raise RuntimeError(
                    'No se pudo descargar el clima después de varios intentos. '
                    'Vuelve a ejecutar esta celda o revisa tu conexión.'
                ) from error

            print(f'Intento {intento} falló al descargar clima: {error}. Reintentando...')
            time.sleep(espera_segundos)


if ruta_clima.exists():
    clima_cdmx = pd.read_csv(ruta_clima)
else:
    parametros = {
        'latitude': 19.4326,
        'longitude': -99.1332,
        'start_date': df_limpio['fecha'].min().strftime('%Y-%m-%d'),
        'end_date': df_limpio['fecha'].max().strftime('%Y-%m-%d'),
        'daily': ','.join(variables_clima),
        'timezone': 'America/Mexico_City',
    }

    url_clima = 'https://archive-api.open-meteo.com/v1/archive?' + urlencode(parametros)

    datos_clima = descargar_json_con_reintentos(url_clima)

    clima_cdmx = pd.DataFrame(datos_clima['daily'])
    clima_cdmx.to_csv(ruta_clima, index=False)

clima_cdmx['fecha'] = pd.to_datetime(clima_cdmx['time'])
clima_cdmx = clima_cdmx.drop(columns=['time'])

print(f'Filas de clima diario: {len(clima_cdmx):,}')
clima_cdmx.head()

In [ ]:
columnas_clima = [col for col in clima_cdmx.columns if col != 'fecha']

df_limpio = df_limpio.drop(columns=columnas_clima, errors='ignore')
df_limpio = df_limpio.merge(clima_cdmx, on='fecha', how='left')

df_limpio['llovio'] = df_limpio['precipitation_sum'].gt(0)
df_limpio['lluvia_fuerte'] = df_limpio['precipitation_sum'].ge(10)

resumen_union_clima = pd.DataFrame({
    'filas_df_limpio': [len(df_limpio)],
    'fechas_unicas_df_limpio': [df_limpio['fecha'].nunique()],
    'fechas_unicas_clima': [clima_cdmx['fecha'].nunique()],
    'filas_sin_clima': [df_limpio[columnas_clima].isna().any(axis=1).sum()],
})

resumen_union_clima

Primero revisamos si el clima se relaciona con la afluencia total diaria. Esto no prueba causalidad, pero ayuda a decidir si vale la pena conservar estas variables para el modelo.

In [ ]:
afluencia_diaria_clima = (
    df_limpio.dropna(subset=['afluencia_modelo'])
    .groupby('fecha')
    .agg(
        afluencia_diaria=('afluencia_modelo', 'sum'),
        temperature_2m_mean=('temperature_2m_mean', 'first'),
        temperature_2m_max=('temperature_2m_max', 'first'),
        temperature_2m_min=('temperature_2m_min', 'first'),
        precipitation_sum=('precipitation_sum', 'first'),
        rain_sum=('rain_sum', 'first'),
        precipitation_hours=('precipitation_hours', 'first'),
        wind_speed_10m_max=('wind_speed_10m_max', 'first'),
        wind_gusts_10m_max=('wind_gusts_10m_max', 'first'),
        es_festivo=('es_festivo', 'first'),
        es_fin_semana=('es_fin_semana', 'first'),
    )
    .reset_index()
)

afluencia_diaria_clima['llovio'] = afluencia_diaria_clima['precipitation_sum'].gt(0)
afluencia_diaria_clima['lluvia_fuerte'] = afluencia_diaria_clima['precipitation_sum'].ge(10)

correlacion_clima_afluencia = (
    afluencia_diaria_clima[
        ['afluencia_diaria'] + variables_clima[:-1]
    ]
    .corr(numeric_only=True)['afluencia_diaria']
    .drop('afluencia_diaria')
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
)

correlacion_clima_afluencia.columns = ['variable_clima', 'correlacion_con_afluencia_diaria']
correlacion_clima_afluencia

In [ ]:
resumen_lluvia_afluencia = (
    afluencia_diaria_clima.groupby('llovio')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

resumen_lluvia_afluencia['tipo_dia'] = resumen_lluvia_afluencia['llovio'].map({False: 'Sin lluvia', True: 'Con lluvia'})
resumen_lluvia_afluencia[['mean', 'median', 'min', 'max']] = resumen_lluvia_afluencia[['mean', 'median', 'min', 'max']].round(0)

resumen_lluvia_afluencia[['tipo_dia', 'mean', 'median', 'min', 'max', 'count']]

In [ ]:
resumen_lluvia_fuerte_afluencia = (
    afluencia_diaria_clima.groupby('lluvia_fuerte')['afluencia_diaria']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

resumen_lluvia_fuerte_afluencia['tipo_dia'] = resumen_lluvia_fuerte_afluencia['lluvia_fuerte'].map({False: 'Sin lluvia fuerte', True: 'Lluvia >= 10 mm'})
resumen_lluvia_fuerte_afluencia[['mean', 'median', 'min', 'max']] = resumen_lluvia_fuerte_afluencia[['mean', 'median', 'min', 'max']].round(0)

resumen_lluvia_fuerte_afluencia[['tipo_dia', 'mean', 'median', 'min', 'max', 'count']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    afluencia_diaria_clima['precipitation_sum'],
    afluencia_diaria_clima['afluencia_diaria'],
    alpha=0.35,
)
axes[0].set_title('Afluencia diaria vs precipitación')
axes[0].set_xlabel('Precipitación diaria (mm)')
axes[0].set_ylabel('Afluencia diaria')

axes[1].scatter(
    afluencia_diaria_clima['temperature_2m_mean'],
    afluencia_diaria_clima['afluencia_diaria'],
    alpha=0.35,
)
axes[1].set_title('Afluencia diaria vs temperatura media')
axes[1].set_xlabel('Temperatura media diaria (°C)')
axes[1].set_ylabel('Afluencia diaria')

plt.tight_layout()
plt.show()

Finalmente revisamos si las estaciones atípicas de media/mediana también tienen picos en días de lluvia. Esto ayuda a distinguir entre picos por clima y picos por eventos locales.

In [ ]:
afluencia_estacion_dia_clima = afluencia_estacion_dia.merge(
    clima_cdmx,
    on='fecha',
    how='left',
)
afluencia_estacion_dia_clima['llovio'] = afluencia_estacion_dia_clima['precipitation_sum'].gt(0)
afluencia_estacion_dia_clima['lluvia_fuerte'] = afluencia_estacion_dia_clima['precipitation_sum'].ge(10)

afluencia_estacion_dia_clima_extremas = afluencia_estacion_dia_clima.merge(
    estaciones_extremas_media_mediana,
    on=['linea', 'estacion'],
    how='inner',
)

comparacion_clima_estaciones_extremas = (
    afluencia_estacion_dia_clima_extremas.groupby(['tipo_diferencia', 'linea', 'estacion', 'lluvia_fuerte'])['afluencia_estacion_dia']
    .agg(['mean', 'median', 'min', 'max', 'count'])
    .reset_index()
)

comparacion_clima_estaciones_extremas['tipo_lluvia'] = comparacion_clima_estaciones_extremas['lluvia_fuerte'].map({False: 'Sin lluvia fuerte', True: 'Lluvia >= 10 mm'})
comparacion_clima_estaciones_extremas[['mean', 'median', 'min', 'max']] = comparacion_clima_estaciones_extremas[['mean', 'median', 'min', 'max']].round(0)

comparacion_clima_estaciones_extremas.sort_values(['tipo_diferencia', 'linea', 'estacion', 'lluvia_fuerte'])

In [ ]:
top_dias_lluvia = afluencia_diaria_clima.sort_values('precipitation_sum', ascending=False).head(20)
top_dias_lluvia[['fecha', 'afluencia_diaria', 'precipitation_sum', 'precipitation_hours', 'temperature_2m_mean', 'es_festivo', 'es_fin_semana']]

**Lectura inicial:** si las correlaciones con clima son bajas, eso no significa que el clima sea inútil: puede aportar en modelos no lineales o en interacción con día de semana, festivos y línea. Para el reporte se debe documentar si la señal es fuerte, débil o principalmente indirecta.

## 19. Dataset transformado y decisión sobre ceros

Varias columnas creadas durante el ETL sirven para **auditoría** y otras para **modelado**.

Decisión sobre los ceros:

- Los ceros parciales por `tipo_pago` se conservan como 0 porque la estación sí tuvo afluencia ese día.
- Los ceros totales de estación-día no se interpretan como demanda baja. Se consideran valores no comparables o ausentes para modelado.
- No se borran del dataset transformado, porque son evidencia de calidad de datos y sirven para el experimento de structural breaks.
- Para entrenar modelos se usa `afluencia_modelo`, donde esos ceros totales ya quedaron como `NaN`, y se crea una versión modelable sin esos registros.

Esto permite tener dos capas: una base transformada completa y una base lista para modelado.

In [ ]:
columnas_transformadas = [
    # Llave y columnas originales limpias
    'fecha', 'mes', 'anio', 'linea', 'estacion', 'tipo_pago', 'afluencia',

    # Target transformado por decisiones ETL
    'afluencia_modelo',

    # Banderas de calidad / structural breaks
    'afluencia_total_dia',
    'cero_total_estacion_dia',
    'cero_parcial_tipo_pago',
    'cero_total_sin_evento',
    'structural_break_metro',
    'evento_estructural',

    # Features calendario
    'dia_semana_num',
    'dia_semana',
    'es_fin_semana',
    'numero_mes',
    'nombre_mes',
    'anio_mes',
    'es_festivo',
    'festivo',

    # Features clima
    'temperature_2m_mean',
    'temperature_2m_max',
    'temperature_2m_min',
    'precipitation_sum',
    'rain_sum',
    'precipitation_hours',
    'wind_speed_10m_max',
    'wind_gusts_10m_max',
    'weather_code',
    'llovio',
    'lluvia_fuerte',
]

columnas_transformadas = [col for col in columnas_transformadas if col in df_limpio.columns]

df_transformado = df_limpio[columnas_transformadas].copy()
df_modelo_desglosado = df_transformado.dropna(subset=['afluencia_modelo']).copy()

resumen_dataset_transformado = pd.DataFrame({
    'dataset': ['df_transformado', 'df_modelo_desglosado'],
    'descripcion': [
        'Base completa con auditoría y decisiones ETL',
        'Base sin ceros totales no modelables, lista para modelar por tipo_pago',
    ],
    'filas': [len(df_transformado), len(df_modelo_desglosado)],
    'columnas': [df_transformado.shape[1], df_modelo_desglosado.shape[1]],
    'nulos_afluencia_modelo': [
        df_transformado['afluencia_modelo'].isna().sum(),
        df_modelo_desglosado['afluencia_modelo'].isna().sum(),
    ],
})

resumen_dataset_transformado

Además del dataset desglosado por `tipo_pago`, creamos una versión agregada por estación-día. Esta versión predice demanda total de la estación y evita que el modelo aprenda solo la distribución contable entre tipos de pago.

In [ ]:
columnas_agregacion = [
    'fecha', 'linea', 'estacion',
    'dia_semana_num', 'dia_semana', 'es_fin_semana', 'numero_mes', 'nombre_mes', 'anio_mes',
    'es_festivo', 'festivo',
    'structural_break_metro', 'evento_estructural', 'cero_total_estacion_dia', 'cero_total_sin_evento',
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'precipitation_sum', 'rain_sum', 'precipitation_hours',
    'wind_speed_10m_max', 'wind_gusts_10m_max', 'weather_code', 'llovio', 'lluvia_fuerte',
]
columnas_agregacion = [col for col in columnas_agregacion if col in df_modelo_desglosado.columns]

df_modelo_desglosado_para_agregar = df_modelo_desglosado.copy()

if 'evento_estructural' in df_modelo_desglosado_para_agregar.columns:
    df_modelo_desglosado_para_agregar['evento_estructural'] = (
        df_modelo_desglosado_para_agregar['evento_estructural'].fillna('Sin evento')
    )

if 'nombre_mes' in df_modelo_desglosado_para_agregar.columns:
    df_modelo_desglosado_para_agregar['nombre_mes'] = df_modelo_desglosado_para_agregar['nombre_mes'].astype(str)

df_modelo_agregado_estacion = (
    df_modelo_desglosado_para_agregar.pivot_table(
        index=columnas_agregacion,
        columns='tipo_pago',
        values='afluencia_modelo',
        aggfunc='sum',
        fill_value=0,
        observed=True,
    )
    .reset_index()
)

df_modelo_agregado_estacion.columns.name = None
df_modelo_agregado_estacion = df_modelo_agregado_estacion.rename(columns={
    'Boleto': 'afluencia_boleto',
    'Prepago': 'afluencia_prepago',
    'Gratuidad': 'afluencia_gratuidad',
})

for columna_pago in ['afluencia_boleto', 'afluencia_prepago', 'afluencia_gratuidad']:
    if columna_pago not in df_modelo_agregado_estacion.columns:
        df_modelo_agregado_estacion[columna_pago] = 0

df_modelo_agregado_estacion['afluencia_total_modelo'] = df_modelo_agregado_estacion[
    ['afluencia_boleto', 'afluencia_prepago', 'afluencia_gratuidad']
].sum(axis=1)

resumen_modelo_agregado = pd.DataFrame({
    'dataset': ['df_modelo_agregado_estacion'],
    'descripcion': ['Base agregada por fecha + linea + estacion'],
    'filas': [len(df_modelo_agregado_estacion)],
    'columnas': [df_modelo_agregado_estacion.shape[1]],
    'target': ['afluencia_total_modelo'],
})

resumen_modelo_agregado

Guardamos los datasets transformados. El PDF pide reportar el tamaño antes y después.

In [ ]:
ruta_transformado = ruta_csv.parent / 'metro_cdmx_transformado.csv'
ruta_modelo_desglosado = ruta_csv.parent / 'metro_cdmx_modelo_desglosado.csv'
ruta_modelo_agregado = ruta_csv.parent / 'metro_cdmx_modelo_agregado_estacion.csv'

df_transformado.to_csv(ruta_transformado, index=False)
df_modelo_desglosado.to_csv(ruta_modelo_desglosado, index=False)
df_modelo_agregado_estacion.to_csv(ruta_modelo_agregado, index=False)

tamanios_archivos = pd.DataFrame([
    {
        'archivo': ruta_csv.name,
        'tipo': 'Original',
        'filas': len(df),
        'columnas': df.shape[1],
        'tamano_mb': ruta_csv.stat().st_size / (1024 ** 2),
    },
    {
        'archivo': ruta_transformado.name,
        'tipo': 'Transformado completo',
        'filas': len(df_transformado),
        'columnas': df_transformado.shape[1],
        'tamano_mb': ruta_transformado.stat().st_size / (1024 ** 2),
    },
    {
        'archivo': ruta_modelo_desglosado.name,
        'tipo': 'Modelado desglosado',
        'filas': len(df_modelo_desglosado),
        'columnas': df_modelo_desglosado.shape[1],
        'tamano_mb': ruta_modelo_desglosado.stat().st_size / (1024 ** 2),
    },
    {
        'archivo': ruta_modelo_agregado.name,
        'tipo': 'Modelado agregado estación-día',
        'filas': len(df_modelo_agregado_estacion),
        'columnas': df_modelo_agregado_estacion.shape[1],
        'tamano_mb': ruta_modelo_agregado.stat().st_size / (1024 ** 2),
    },
])

tamanios_archivos['tamano_mb'] = tamanios_archivos['tamano_mb'].round(2)
tamanios_archivos

**Decisión para los datos con ceros totales:** no se eliminan del archivo transformado completo. Se conservan con banderas para documentación y experimentos. Para modelado, quedan excluidos de `df_modelo_desglosado` y `df_modelo_agregado_estacion` porque `afluencia_modelo` los convirtió en ausentes antes de construir esas bases.

## 20. Preparación para modelado: target, split, encoding y scaling

Antes de entrenar modelos se revisa el target y se preparan transformaciones de variables. Muy importante: cualquier transformación que aprenda parámetros de los datos debe ajustarse **solo con train** y después aplicarse a test.

Para el modelo principal usaremos la base agregada por `fecha + linea + estacion`, porque predice demanda total de la estación y evita que el modelo aprenda solo relaciones contables entre tipos de pago.

Target principal: `afluencia_total_modelo`.

In [ ]:
df_modelado = df_modelo_agregado_estacion.copy()

target_modelo = 'afluencia_total_modelo'

columnas_fuga_target = [
    'afluencia_boleto',
    'afluencia_prepago',
    'afluencia_gratuidad',
    target_modelo,
]

print(f'Filas para modelado: {len(df_modelado):,}')
print(f'Columnas disponibles: {df_modelado.shape[1]:,}')
print('Columnas que NO deben usarse como predictores por fuga del target:')
display(columnas_fuga_target)

df_modelado.head()

### 20.1 Features temporales con memoria histórica

Además de calendario, agregamos variables que le dicen al modelo cómo venía comportándose cada estación en el pasado.

- `time_idx`: número de días desde el inicio de la base.
- `lag_7`: afluencia de la misma estación hace 7 días.
- `lag_14`: afluencia de la misma estación hace 14 días.
- `rolling_mean_7`: promedio de los 7 días previos.
- `rolling_mean_28`: promedio de los 28 días previos.

Estas variables se calculan con `shift`, por lo que usan solo información pasada. No usan datos del futuro.

In [ ]:
df_modelado = df_modelado.sort_values(['linea', 'estacion', 'fecha']).copy()

df_modelado['time_idx'] = (
    df_modelado['fecha'] - df_modelado['fecha'].min()
).dt.days

grupo_estacion = df_modelado.groupby(['linea', 'estacion'])[target_modelo]

df_modelado['lag_7'] = grupo_estacion.shift(7)
df_modelado['lag_14'] = grupo_estacion.shift(14)
df_modelado['lag_28'] = grupo_estacion.shift(28)

df_modelado['rolling_mean_7'] = grupo_estacion.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=3).mean()
)
df_modelado['rolling_mean_28'] = grupo_estacion.transform(
    lambda s: s.shift(1).rolling(window=28, min_periods=7).mean()
)

features_temporales = ['time_idx', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']

resumen_features_temporales = pd.DataFrame({
    'feature': features_temporales,
    'nulos': [df_modelado[col].isna().sum() for col in features_temporales],
    'pct_nulos': [(df_modelado[col].isna().mean() * 100).round(2) for col in features_temporales],
})

resumen_features_temporales

In [ ]:
df_modelado_lags = df_modelado.dropna(subset=['lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28']).copy()

resumen_modelado_lags = pd.DataFrame({
    'dataset': ['df_modelado', 'df_modelado_lags'],
    'descripcion': [
        'Base agregada antes de quitar nulos de lags',
        'Base agregada con features temporales completas',
    ],
    'filas': [len(df_modelado), len(df_modelado_lags)],
    'fecha_min': [df_modelado['fecha'].min(), df_modelado_lags['fecha'].min()],
    'fecha_max': [df_modelado['fecha'].max(), df_modelado_lags['fecha'].max()],
})

resumen_modelado_lags

### 20.2 Revisión del target

El PDF menciona que el target suele estar sesgado a la derecha. Eso significa que hay muchas observaciones bajas o medias y pocas observaciones extremadamente altas. Revisamos la distribución original y la versión `log1p`.

In [ ]:
target_resumen = df_modelado[target_modelo].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame(name=target_modelo)

target_resumen.loc['skewness'] = df_modelado[target_modelo].skew()
target_resumen.round(2)

In [ ]:
import numpy as np

df_modelado['target_log1p'] = np.log1p(df_modelado[target_modelo])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_modelado[target_modelo], bins=60)
axes[0].set_title('Distribución del target original')
axes[0].set_xlabel('Afluencia total diaria')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(df_modelado['target_log1p'], bins=60)
axes[1].set_title('Distribución del target con log1p')
axes[1].set_xlabel('log1p(afluencia total diaria)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

pd.DataFrame({
    'target': ['original', 'log1p'],
    'skewness': [df_modelado[target_modelo].skew(), df_modelado['target_log1p'].skew()],
}).round(3)

**Decisión preliminar:** se conserva el target original para interpretación y también se prepara `log1p`. Si se entrena un modelo con `log1p`, las predicciones deben regresar a escala real con `expm1` antes de reportar MAE/RMSE en usuarios.

### 20.3 Split cronológico

Como es una serie de tiempo, el split no debe ser aleatorio para el modelo principal. Usamos un corte cronológico: train hasta `2025-06-30` y test desde `2025-07-01` hasta `2026-06-30`.

In [ ]:
fecha_corte_test = pd.Timestamp('2025-07-01')

train_mask = df_modelado_lags['fecha'].lt(fecha_corte_test)
test_mask = df_modelado_lags['fecha'].ge(fecha_corte_test)

df_train = df_modelado_lags[train_mask].copy()
df_test = df_modelado_lags[test_mask].copy()

resumen_split_cronologico = pd.DataFrame({
    'split': ['train', 'test'],
    'fecha_min': [df_train['fecha'].min(), df_test['fecha'].min()],
    'fecha_max': [df_train['fecha'].max(), df_test['fecha'].max()],
    'filas': [len(df_train), len(df_test)],
    'estaciones': [df_train[['linea', 'estacion']].drop_duplicates().shape[0], df_test[['linea', 'estacion']].drop_duplicates().shape[0]],
})

resumen_split_cronologico

### 20.4 Columnas predictoras candidatas

Separamos variables categóricas y numéricas. No usamos columnas como `afluencia_boleto`, `afluencia_prepago` o `afluencia_gratuidad` como features porque son partes del target total y causarían fuga de información.

In [ ]:
features_categoricas = [
    'linea',
    'estacion',
    'dia_semana',
    'festivo',
    'evento_estructural',
]

features_numericas = [
    'time_idx',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'dia_semana_num',
    'numero_mes',
    'es_fin_semana',
    'es_festivo',
    'structural_break_metro',
    'temperature_2m_mean',
    'temperature_2m_max',
    'temperature_2m_min',
    'precipitation_sum',
    'rain_sum',
    'precipitation_hours',
    'wind_speed_10m_max',
    'wind_gusts_10m_max',
    'llovio',
    'lluvia_fuerte',
]

features_categoricas = [col for col in features_categoricas if col in df_modelado.columns]
features_numericas = [col for col in features_numericas if col in df_modelado.columns]

pd.DataFrame({
    'tipo_feature': ['categoricas', 'numericas'],
    'columnas': [features_categoricas, features_numericas],
    'n_columnas': [len(features_categoricas), len(features_numericas)],
})

### 20.5 Comparación de encoding

Comparamos dos estrategias:

- **One-hot encoding:** más interpretable, pero crea muchas columnas, especialmente por `estacion`.
- **Frequency encoding:** usa poca memoria porque reemplaza cada categoría por su frecuencia en train, pero pierde detalle categórico.

Ambas se ajustan usando solo train.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

X_train_onehot = onehot_encoder.fit_transform(df_train[features_categoricas])
X_test_onehot = onehot_encoder.transform(df_test[features_categoricas])

onehot_feature_names = onehot_encoder.get_feature_names_out(features_categoricas)

memoria_onehot_train_mb = (
    X_train_onehot.data.nbytes
    + X_train_onehot.indptr.nbytes
    + X_train_onehot.indices.nbytes
) / (1024 ** 2)

resumen_onehot = pd.DataFrame({
    'encoding': ['one_hot'],
    'filas_train': [X_train_onehot.shape[0]],
    'columnas_generadas': [X_train_onehot.shape[1]],
    'memoria_train_mb_aprox': [round(memoria_onehot_train_mb, 2)],
    'categorias_aprendidas_solo_train': [len(onehot_feature_names)],
})

resumen_onehot

In [ ]:
def ajustar_frequency_encoding(train, columnas):
    mapas = {}
    for columna in columnas:
        mapas[columna] = train[columna].value_counts(normalize=True, dropna=False)
    return mapas

def aplicar_frequency_encoding(df_base, mapas):
    df_encoded = pd.DataFrame(index=df_base.index)
    for columna, mapa in mapas.items():
        df_encoded[f'{columna}_freq'] = df_base[columna].map(mapa).fillna(0)
    return df_encoded

mapas_frequency_encoding = ajustar_frequency_encoding(df_train, features_categoricas)
X_train_freq = aplicar_frequency_encoding(df_train, mapas_frequency_encoding)
X_test_freq = aplicar_frequency_encoding(df_test, mapas_frequency_encoding)

memoria_freq_train_mb = X_train_freq.memory_usage(deep=True).sum() / (1024 ** 2)

resumen_frequency = pd.DataFrame({
    'encoding': ['frequency'],
    'filas_train': [X_train_freq.shape[0]],
    'columnas_generadas': [X_train_freq.shape[1]],
    'memoria_train_mb_aprox': [round(memoria_freq_train_mb, 2)],
    'categorias_aprendidas_solo_train': [sum(len(mapa) for mapa in mapas_frequency_encoding.values())],
})

pd.concat([resumen_onehot, resumen_frequency], ignore_index=True)

**Decisión preliminar de encoding:** para un modelo lineal conviene `one-hot` porque permite efectos separados por estación y línea. Para modelos simples o con restricciones de memoria, `frequency encoding` es más liviano, pero pierde interpretabilidad y puede subrepresentar estaciones importantes.

### 20.6 Scaling de variables numéricas

El escalamiento se ajusta solo con train. Esto evita que información del futuro entre al preprocesamiento. Los modelos lineales suelen beneficiarse de scaling; los modelos de árboles no lo necesitan tanto.

In [ ]:
scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(df_train[features_numericas]),
    columns=features_numericas,
    index=df_train.index,
)

X_test_num_scaled = pd.DataFrame(
    scaler.transform(df_test[features_numericas]),
    columns=features_numericas,
    index=df_test.index,
)

revision_scaling = pd.DataFrame({
    'feature': features_numericas,
    'media_train_original': df_train[features_numericas].mean(numeric_only=True).values,
    'std_train_original': df_train[features_numericas].std(numeric_only=True).values,
    'media_train_escalada': X_train_num_scaled.mean().values,
    'std_train_escalada': X_train_num_scaled.std().values,
})

revision_scaling.round(3)

Con esto queda preparado el preprocesamiento. El siguiente paso ya sería entrenar modelos usando el split cronológico: primero baseline, luego un modelo lineal y después un ensamble basado en árboles.